In [ ]:
import os
import gc
import pickle
import re
import numpy as np
import tensorflow as tf

from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import pickle

DATA_PATH = "/kaggle/input/datasets/suyashma/math-expression1/processed_crohme.pkl"
if not os.path.exists(DATA_PATH):
    base_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    local_data = os.path.join(base_dir, "dataset", "processed_crohme.pkl")
    if os.path.exists(local_data):
        DATA_PATH = local_data

with open(DATA_PATH, "rb") as f:
    processed_data, skipped_files = pickle.load(f)

data = processed_data

print("Processed samples:", len(processed_data))
print("Skipped files:", len(skipped_files))

In [ ]:
print(type(processed_data))
print(len(processed_data))

print(type(processed_data[0]))
print(len(processed_data[0]))

In [ ]:
import re
def tokenize_latex(expression):
    expression=expression.replace("$","")
    tokens=re.findall(
         r'\\[a-zA-Z]+|[0-9]+|[a-zA-Z]|[^\s]',
        expression
    )

    return tokens

In [ ]:
test_labels = [
    r"$\phi(x)$",
    r"$x^2$",
    r"$\frac{x}{2}$"
]

for label in test_labels:
    print(label, "→", tokenize_latex(label))

In [ ]:
from collections import Counter

token_counter = Counter()

for sample in data:
    tokens = tokenize_latex(sample["label"])
    token_counter.update(tokens)

print("Total unique tokens:", len(token_counter))

print("\nMost common tokens:")
for token, count in token_counter.most_common(30):
    print(token, ":", count)

special_tokens = [
    "<pad>",
    "<unk>"
]

unique_tokens = sorted(token_counter.keys())

# <blank> must be the LAST token for ctc_batch_cost()
vocabulary = special_tokens + unique_tokens + ["<blank>"]

token_to_id = {
    token: i
    for i, token in enumerate(vocabulary)
}

id_to_token = {
    i: token
    for token, i in token_to_id.items()
}

print("\nVocabulary size:", len(vocabulary))
print("PAD ID:", token_to_id["<pad>"])
print("UNK ID:", token_to_id["<unk>"])
print("BLANK ID:", token_to_id["<blank>"])
print("Last vocabulary ID:", len(vocabulary) - 1)

In [ ]:
token_to_id = {
    token: i
    for i, token in enumerate(vocabulary)
}

id_to_token = {
    i: token
    for i, token in enumerate(vocabulary)
}

print("PAD:", token_to_id["<pad>"])
print("UNK:", token_to_id["<unk>"])
print("BLANK:", token_to_id["<blank>"])

In [ ]:
def encode_label(label):

    tokens = tokenize_latex(label)

    encoded = []

    for token in tokens:

        if token in token_to_id:
            encoded.append(token_to_id[token])
        else:
            encoded.append(token_to_id["<unk>"])

    return encoded

In [ ]:
sample_label=data[0]["label"]
tokens = tokenize_latex(sample_label)
encoded = encode_label(sample_label)

print("Original label:")
print(sample_label)

print("\nTokens:")
print(tokens)

print("\nEncoded:")
print(encoded)

In [ ]:
import numpy as np
def strokes_to_sequence(strokes):
    sequence=[]
    for stoke_index,stroke in enumerate(strokes):
        for point_index,(x,y) in enumerate(stroke):
            pen=1 if point_index==0 else 0
            sequence.append([x,y,pen])

    return np.array(sequence, dtype=np.float32)

In [ ]:
sample_strokes = data[0]["strokes"]

sequence = strokes_to_sequence(sample_strokes)

print("Sequence shape:", sequence.shape)
print("\nFirst 10 points:")
print(sequence[:10])

In [ ]:
X = []
y = []

for sample in data:

    sequence = strokes_to_sequence(sample["strokes"])
    label = encode_label(sample["label"])

    X.append(sequence)
    y.append(label)

print("Total samples:", len(X))
print("First encoded label:", y[0])
print("First original label:", data[0]["label"])

In [ ]:
print("Input shape:", X[0].shape)
print("Target:", y[0])
print("Target length:", len(y[0]))

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

In [ ]:
max_length = max(len(sequence) for sequence in X_train)

print("Maximum sequence length:", max_length)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_train_padded=pad_sequences(
    X_train,
    maxlen=max_length,
    padding="post",
    dtype="float32"
)
X_val_padded=pad_sequences(
    X_val,
    maxlen=max_length,
    padding="post",
    dtype="float32"
)
print("Training shape:", X_train_padded.shape)
print("Validation shape:", X_val_padded.shape)

In [ ]:
max_label_length = max(len(label) for label in y_train)

print("Maximum label length:", max_label_length)

In [ ]:
max_label_length = max(
    len(label)
    for label in y_train
)

print("Maximum label length:", max_label_length)

label_lengths_train = np.array(
    [len(label) for label in y_train],
    dtype=np.int32
)

label_lengths_val = np.array(
    [len(label) for label in y_val],
    dtype=np.int32
)

print("Train label lengths:", label_lengths_train.shape)
print("Val label lengths:", label_lengths_val.shape)

In [ ]:
import numpy as np

y_train_padded = np.full(
    (len(y_train), max_label_length),
    token_to_id["<pad>"],
    dtype=np.int32
)

y_val_padded = np.full(
    (len(y_val), max_label_length),
    token_to_id["<pad>"],
    dtype=np.int32
)

for i, label in enumerate(y_train):
    y_train_padded[i, :len(label)] = label

for i, label in enumerate(y_val):
    y_val_padded[i, :len(label)] = label

print("Training targets:", y_train_padded.shape)
print("Validation targets:", y_val_padded.shape)

In [ ]:
input_lengths_train = np.array(
    [len(sequence) for sequence in X_train],
    dtype=np.int32
)

input_lengths_val = np.array(
    [len(sequence) for sequence in X_val],
    dtype=np.int32
)

print("Train input lengths:")
print(input_lengths_train.min())
print(input_lengths_train.max())

print("\nValidation input lengths:")
print(input_lengths_val.min())
print(input_lengths_val.max())

In [ ]:
input_lengths_train_new = (
    input_lengths_train // 2
).astype(np.int32)

input_lengths_val_new = (
    input_lengths_val // 2
).astype(np.int32)

input_lengths_train_ctc = (
    input_lengths_train_new.reshape(-1, 1)
)

input_lengths_val_ctc = (
    input_lengths_val_new.reshape(-1, 1)
)

label_lengths_train_ctc = (
    label_lengths_train.reshape(-1, 1)
)

label_lengths_val_ctc = (
    label_lengths_val.reshape(-1, 1)
)

print("Train CTC input:", input_lengths_train_ctc.shape)
print("Val CTC input:", input_lengths_val_ctc.shape)

In [ ]:
y_train_ctc = y_train_padded.copy()
y_val_ctc = y_val_padded.copy()

# Replace padding -1 with 0.
# label_length tells CTC which positions are real.
y_train_ctc[y_train_ctc == -1] = 0
y_val_ctc[y_val_ctc == -1] = 0

print("Train CTC labels:", y_train_ctc.shape)
print("Val CTC labels:", y_val_ctc.shape)

In [ ]:
def ctc_required_length(label):

    required = len(label)

    for i in range(1, len(label)):

        if label[i] == label[i - 1]:
            required += 1

    return required


invalid_train = []

for i, label in enumerate(y_train):

    required = ctc_required_length(label)

    available = input_lengths_train_new[i]

    if required > available:
        invalid_train.append(
            (i, available, required)
        )


invalid_val = []

for i, label in enumerate(y_val):

    required = ctc_required_length(label)

    available = input_lengths_val_new[i]

    if required > available:
        invalid_val.append(
            (i, available, required)
        )


print("CTC-invalid training samples:", len(invalid_train))
print("CTC-invalid validation samples:", len(invalid_val))

In [ ]:
import tensorflow as tf 
import tensorflow as tf
from tensorflow.keras import layers, Model

# ============================================
# 1. Stroke Input
# ============================================

stroke_input_new = layers.Input(
    shape=(max_length, 3),
    name="stroke_input"
)


# ============================================
# 2. Conv1D Feature Extraction
# ============================================

x = layers.Conv1D(
    filters=64,
    kernel_size=5,
    padding="same",
    activation="relu"
)(stroke_input_new)


# ============================================
# 3. Downsample Sequence
# ============================================

x = layers.MaxPooling1D(
    pool_size=2
)(x)


# ============================================
# 4. BiLSTM Sequence Modeling
# ============================================

x = layers.Bidirectional(
    layers.LSTM(
        128,
        return_sequences=True
    )
)(x)

x = layers.Bidirectional(
    layers.LSTM(
        128,
        return_sequences=True
    )
)(x)


# ============================================
# 5. Token Prediction
# ============================================

token_output_new = layers.Dense(
    len(vocabulary),
    activation="softmax",
    name="token_output"
)(x)


# ============================================
# 6. CTC Inputs
# ============================================

labels_new = layers.Input(
    shape=(max_label_length,),
    dtype="int32",
    name="labels"
)

input_length_new = layers.Input(
    shape=(1,),
    dtype="int32",
    name="input_length"
)

label_length_new = layers.Input(
    shape=(1,),
    dtype="int32",
    name="label_length"
)

In [ ]:
labels_new = layers.Input(
    shape=(max_label_length,),
    dtype="int32",
    name="labels"
)

input_length_new = layers.Input(
    shape=(1,),
    dtype="int32",
    name="input_length"
)

label_length_new = layers.Input(
    shape=(1,),
    dtype="int32",
    name="label_length"
)


def ctc_loss_function_new(args):

    labels, y_pred, input_length, label_length = args

    return tf.keras.backend.ctc_batch_cost(
        labels,
        y_pred,
        input_length,
        label_length
    )


loss_output_new = layers.Lambda(
    ctc_loss_function_new,
    output_shape=(1,),
    name="ctc_loss"
)(
    [
        labels_new,
        token_output_new,
        input_length_new,
        label_length_new
    ]
)


ctc_model_new = Model(
    inputs=[
        stroke_input_new,
        labels_new,
        input_length_new,
        label_length_new
    ],
    outputs=loss_output_new
)


ctc_model_new.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=lambda y_true, y_pred: y_pred
)


ctc_model_new.summary()

In [ ]:
train_inputs_new = {

    "stroke_input": X_train_padded,

    "labels": y_train_ctc,

    "input_length": input_lengths_train_ctc,

    "label_length": label_lengths_train_ctc
}


val_inputs_new = {

    "stroke_input": X_val_padded,

    "labels": y_val_ctc,

    "input_length": input_lengths_val_ctc,

    "label_length": label_lengths_val_ctc
}


train_dummy_new = np.zeros(
    (len(X_train), 1),
    dtype=np.float32
)

val_dummy_new = np.zeros(
    (len(X_val), 1),
    dtype=np.float32
)


print("Training ready.")

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_path = (
    "/kaggle/working/mathinkai_epoch_{epoch:02d}.weights.h5"
)

checkpoint_callback = ModelCheckpoint(
    checkpoint_path,
    save_weights_only=True,
    save_freq="epoch",
    verbose=1
)

In [ ]:
history_new = ctc_model_new.fit(

    train_inputs_new,

    train_dummy_new,

    validation_data=(
        val_inputs_new,
        val_dummy_new
    ),

    batch_size=16,

    epochs=10,
    callbacks=[
        checkpoint_callback
    ],

    verbose=1
)

In [ ]:
inference_model_new = Model(
    inputs=stroke_input_new,
    outputs=token_output_new
)

print("Inference model created.")

In [ ]:
N = 10

predictions_new = inference_model_new.predict(
    X_val_padded[:N],
    batch_size=1,
    verbose=1
)

print("Prediction shape:", predictions_new.shape)

In [ ]:
def greedy_ctc_decode(
    prediction,
    input_length
):

    prediction = prediction[
        :int(input_length)
    ]

    token_ids = np.argmax(
        prediction,
        axis=-1
    )

    blank_id = token_to_id["<blank>"]

    decoded = []

    previous_token = None

    for token_id in token_ids:

        token_id = int(token_id)

        if token_id == blank_id:
            previous_token = None
            continue

        if token_id == previous_token:
            continue

        token = id_to_token[token_id]

        decoded.append(token)

        previous_token = token_id

    return decoded

In [ ]:
def ids_to_tokens(label_ids):
    
    tokens = []
    
    for token_id in label_ids:
        
        token_id = int(token_id)
        
        if token_id == token_to_id["<pad>"]:
            continue
        
        tokens.append(id_to_token[token_id])
    
    return tokens

In [ ]:
decoded_sequence_new = []

for i in range(N):

    decoded = greedy_ctc_decode(
        predictions_new[i],
        input_lengths_val_new[i]
    )

    decoded_sequence_new.append(decoded)

In [ ]:
for i in range(5):

    print("\n==============================")
    print("Sample", i + 1)
    print("==============================")

    print("Actual:")
    print(ids_to_tokens(y_val[i]))

    print("Predicted:")
    print(decoded_sequence_new[i])

In [ ]:
correct=0
total=N
for i in range(N):
    actual=ids_to_tokens(y_val[i])
    predicted=decoded_sequence_new[i]
    if actual==predicted:
        correct+=1

accuracy = correct / total

print(f"Exact sequence accuracy: {accuracy * 100:.2f}%")

In [ ]:
total_tokens = 0
correct_tokens = 0

for i in range(N):
    actual = ids_to_tokens(y_val[i])
    predicted = decoded_sequence_new[i]

    # Compare only positions that exist in both sequences
    compare_length = min(len(actual), len(predicted))

    for j in range(compare_length):
        if actual[j] == predicted[j]:
            correct_tokens += 1

    # Count ALL actual tokens as the denominator
    total_tokens += len(actual)

token_accuracy = correct_tokens / total_tokens

print("Correct tokens:", correct_tokens)
print("Total actual tokens:", total_tokens)
print(f"Token-level accuracy: {token_accuracy * 100:.2f}%")

In [ ]:
N=300
predictions_100=inference_model_new.predict(
    X_val_padded[:N],
    batch_size=1,
    verbose=1
)
decoded_sequences_100=[]
for i in range(N):
    decoded=greedy_ctc_decode(
        predictions_100[i],
        input_lengths_val_new[i]
    )
    decoded_sequences_100.append(decoded)

print("Predictions completed:", len(decoded_sequences_100))

In [ ]:
total_tokens = 0
correct_tokens = 0
exact_matches = 0

for i in range(N):
    actual = ids_to_tokens(y_val[i])
    predicted = decoded_sequences_100[i]

    # Token-level accuracy
    compare_length = min(len(actual), len(predicted))

    for j in range(compare_length):
        if actual[j] == predicted[j]:
            correct_tokens += 1

    total_tokens += len(actual)

    # Exact sequence accuracy
    if actual == predicted:
        exact_matches += 1

token_accuracy_100 = correct_tokens / total_tokens
sequence_accuracy_100 = exact_matches / N

print("========== EVALUATION ==========")
print("Samples tested:", N)
print("Correct tokens:", correct_tokens)
print("Total actual tokens:", total_tokens)
print(f"Token-level accuracy: {token_accuracy_100 * 100:.2f}%")
print("Exact matches:", exact_matches)
print(f"Exact sequence accuracy: {sequence_accuracy_100 * 100:.2f}%")

In [ ]:
def edit_distance(seq1,seq2):
    m=len(seq1)
    n=len(seq2)
    dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1):
        dp[i][0]=i
    for j in range(n+1):
        dp[0][j]=j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[m][n]

total_distance=0
total_tokens=0
for i in range(N):
    actual = ids_to_tokens(y_val[i])
    predicted = decoded_sequences_200[i] if N == 200 else decoded_sequences_100[i]

    distance = edit_distance(actual, predicted)

    total_distance += distance
    total_tokens += len(actual)

normalized_edit_distance = total_distance / total_tokens
sequence_similarity = 1 - normalized_edit_distance

print("========== EDIT DISTANCE EVALUATION ==========")
print("Samples tested:", N)
print("Total edit distance:", total_distance)
print("Total actual tokens:", total_tokens)
print(f"Normalized edit distance: {normalized_edit_distance:.4f}")
print(f"Sequence similarity: {sequence_similarity * 100:.2f}%")

In [ ]:
for i in range(20):
    actual = ids_to_tokens(y_val[i])
    predicted = decoded_sequences_100[i]

    print(f"\n========== Sample {i+1} ==========")
    print("Actual:   ", " ".join(actual))
    print("Predicted:", " ".join(predicted))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    history_new.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_new.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("CTC Loss")
plt.title("MathInkAI Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
ctc_model_new.save_weights("/kaggle/working/mathinkai_epoch_10.weights.h5")

print("Epoch 10 weights saved successfully.")

In [ ]:
history_continue = ctc_model_new.fit(
    train_inputs_new,
    train_dummy_new,
    validation_data=(val_inputs_new, val_dummy_new),
    batch_size=16,
    epochs=20,
    initial_epoch=10,
    verbose=1
)

In [ ]:
N=300
predictions_100=inference_model_new.predict(
    X_val_padded[:N],
    batch_size=1,
    verbose=1
)
decoded_sequences_100=[]
for i in range(N):
    decoded=greedy_ctc_decode(
        predictions_100[i],
        input_lengths_val_new[i]
    )
    decoded_sequences_100.append(decoded)

print("Predictions completed:", len(decoded_sequences_100))

In [ ]:
total_tokens = 0
correct_tokens = 0
exact_matches = 0

for i in range(N):
    actual = ids_to_tokens(y_val[i])
    predicted = decoded_sequences_100[i]

    # Token-level accuracy
    compare_length = min(len(actual), len(predicted))

    for j in range(compare_length):
        if actual[j] == predicted[j]:
            correct_tokens += 1

    total_tokens += len(actual)

    # Exact sequence accuracy
    if actual == predicted:
        exact_matches += 1

token_accuracy_100 = correct_tokens / total_tokens
sequence_accuracy_100 = exact_matches / N

print("========== EVALUATION ==========")
print("Samples tested:", N)
print("Correct tokens:", correct_tokens)
print("Total actual tokens:", total_tokens)
print(f"Token-level accuracy: {token_accuracy_100 * 100:.2f}%")
print("Exact matches:", exact_matches)
print(f"Exact sequence accuracy: {sequence_accuracy_100 * 100:.2f}%")

In [ ]:
total_distance = 0
total_tokens = 0

for i in range(N):
    actual = ids_to_tokens(y_val[i])
    predicted = decoded_sequences_100[i]

    distance = edit_distance(actual, predicted)

    total_distance += distance
    total_tokens += len(actual)

normalized_edit_distance = total_distance / total_tokens
sequence_similarity = 1 - normalized_edit_distance

print("========== EPOCH 20 EDIT DISTANCE ==========")
print("Samples tested:", N)
print("Total edit distance:", total_distance)
print("Total actual tokens:", total_tokens)
print(f"Normalized edit distance: {normalized_edit_distance:.4f}")
print(f"Sequence similarity: {sequence_similarity * 100:.2f}%")

In [ ]:
ctc_model_new.save_weights(
    "/kaggle/working/mathinkai_epoch_20.weights.h5"
)

print("Epoch 20 weights saved.")

In [ ]:
history_continue_2 = ctc_model_new.fit(
    train_inputs_new,
    train_dummy_new,
    validation_data=(val_inputs_new, val_dummy_new),
    batch_size=16,
    epochs=71,
    initial_epoch=20,
    verbose=1
)

In [ ]:
N=300
predictions_100=inference_model_new.predict(
    X_val_padded[:N],
    batch_size=1,
    verbose=1
)
decoded_sequences_100=[]
for i in range(N):
    decoded=greedy_ctc_decode(
        predictions_100[i],
        input_lengths_val_new[i]
    )
    decoded_sequences_100.append(decoded)

print("Predictions completed:", len(decoded_sequences_100))

In [ ]:
import os

print("Working directory:")
print(os.listdir("/kaggle/working"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
print("\nInput datasets:")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith((".h5", ".keras", ".ckpt")):
            print(os.path.join(root, file))

In [ ]:
N=300
predictions_100=inference_model_new.predict(
    X_val_padded[:N],
    batch_size=1,
    verbose=1
)
decoded_sequences_100=[]
for i in range(N):
    decoded=greedy_ctc_decode(
        predictions_100[i],
        input_lengths_val_new[i]
    )
    decoded_sequences_100.append(decoded)

print("Predictions completed:", len(decoded_sequences_100))

In [ ]:
total_tokens = 0
correct_tokens = 0
exact_matches = 0

for i in range(N):
    actual = ids_to_tokens(y_val[i])
    predicted = decoded_sequences_100[i]

    # Token-level accuracy
    compare_length = min(len(actual), len(predicted))

    for j in range(compare_length):
        if actual[j] == predicted[j]:
            correct_tokens += 1

    total_tokens += len(actual)

    # Exact sequence accuracy
    if actual == predicted:
        exact_matches += 1

token_accuracy_100 = correct_tokens / total_tokens
sequence_accuracy_100 = exact_matches / N

print("========== EVALUATION ==========")
print("Samples tested:", N)
print("Correct tokens:", correct_tokens)
print("Total actual tokens:", total_tokens)
print(f"Token-level accuracy: {token_accuracy_100 * 100:.2f}%")
print("Exact matches:", exact_matches)
print(f"Exact sequence accuracy: {sequence_accuracy_100 * 100:.2f}%")

In [ ]:
base_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
save_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else os.path.join(base_dir, "model")
os.makedirs(save_dir, exist_ok=True)
ctc_model_new.save_weights(os.path.join(save_dir, "mathinkai_epoch_71.weights.h5"))

print("MathInkAI epoch-71 weights saved successfully.")

In [ ]:
# !pip install -q latex2sympy2

In [ ]:
import sympy as sp
from latex2sympy2 import latex2sympy

print("SymPy version:", sp.__version__)
print("LaTeX parser imported successfully")

In [ ]:
solutions = latex2sympy(r"x^2 + 5*x + 6 = 0")

print("Raw result:", solutions)

if isinstance(solutions, list):
    clean_solutions = []

    for solution in solutions:
        if isinstance(solution, sp.Equality):
            clean_solutions.append(solution.rhs)
        else:
            clean_solutions.append(solution)

    print("Clean solutions:", clean_solutions)

In [ ]:
print("Python is working")

In [ ]:
def solve_latex(latex):
    try:
        result = latex2sympy(latex)

        # If parser directly returns a list of solutions
        if isinstance(result, list):
            solutions = []

            for item in result:
                if isinstance(item, sp.Equality):
                    solutions.append(item.rhs)
                else:
                    solutions.append(item)

            return solutions

        # If parser returns an equation
        if isinstance(result, sp.Equality):
            symbols = list(result.free_symbols)

            if symbols:
                return sp.solve(result, symbols[0])

        # Otherwise return the SymPy expression
        return result

    except Exception as e:
        return f"Error: {e}"

In [ ]:
test_expressions = [
    r"x^2 + 5*x + 6 = 0",
    r"x + 5 = 12",
    r"2*x - 7 = 9",
    r"\frac{x}{2} + 3 = 7",
    r"x^2 - 9 = 0"
]

for latex in test_expressions:
    result = solve_latex(latex)

    print("\nLaTeX:", latex)
    print("Solution:", result)

In [ ]:
def tokens_to_latex(tokens):
    """
    Convert MathInkAI predicted token list into a LaTeX string.
    """

    latex = "".join(tokens)

    return latex

In [ ]:
test_tokens = [
    "x", "^", "{", "2", "}", "+", "5", "x", "+", "6", "=", "0"
]

latex = tokens_to_latex(test_tokens)

print("Tokens:")
print(test_tokens)

print("\nGenerated LaTeX:")
print(latex)

In [ ]:
# Select one validation sample
i = 0

predicted_tokens = decoded_sequences_100[i]

# Convert predicted tokens to LaTeX
predicted_latex = tokens_to_latex(predicted_tokens)

print("Predicted tokens:")
print(predicted_tokens)

print("\nPredicted LaTeX:")
print(predicted_latex)

In [ ]:
solution=solve_latex(predicted_latex)
print("\nMathInk.ai")
print(solution)

In [ ]:
i=0
predicted_token=decoded_sequences_100[i]
predicted_latex=tokens_to_latex(predicted_token)
solution=solve_latex(predicted_latex)
actual_tokens=ids_to_tokens(y_val[i])

print("\nActual Tokens:")
print(actual_tokens)

print("\nPredicted Tokens:")
print(predicted_tokens)

print("\nPredicted LaTeX:")
print(predicted_latex)

print("\nSolution:")
print(solution)

In [ ]:
def validate_math_expression(latex):
    """
    Validate the predicted LaTeX expression before solving.
    """

    try:
        # Convert LaTeX to SymPy
        expr = latex2sympy(latex)

        # Check if it is an equation
        if isinstance(expr, sp.Equality):

            # Check for undefined functions
            functions = expr.atoms(sp.Function)

            if functions:
                return {
                    "status": "invalid_for_solving",
                    "reason": "Expression contains undefined functions",
                    "expression": expr
                }

            return {
                "status": "valid_equation",
                "reason": "Valid mathematical equation",
                "expression": expr
            }

        # Normal mathematical expression
        return {
            "status": "valid_expression",
            "reason": "Valid mathematical expression",
            "expression": expr
        }

    except Exception as e:

        return {
            "status": "invalid",
            "reason": str(e),
            "expression": None
        }

In [ ]:
validation=validate_math_expression(predicted_latex)
print("Predicted LaTeX:")
print(predicted_latex)

print("\nStatus:")
print(validation["status"])

print("\nReason:")
print(validation["reason"])

print("\nParsed Expression:")
print(validation["expression"])

In [ ]:
def mathink_solve(latex):

    validation = validate_math_expression(latex)

    # Invalid expression
    if validation["status"] == "invalid":
        return {
            "latex": latex,
            "status": "invalid",
            "solution": None,
            "message": validation["reason"]
        }

    # Cannot solve undefined functions
    if validation["status"] == "invalid_for_solving":
        return {
            "latex": latex,
            "status": "not_solvable",
            "solution": None,
            "message": validation["reason"]
        }

    # Valid equation → solve
    if validation["status"] == "valid_equation":

        try:
            expr = validation["expression"]

            symbols = list(expr.free_symbols)

            if len(symbols) == 0:
                return {
                    "latex": latex,
                    "status": "valid",
                    "solution": expr,
                    "message": "Equation contains no variable."
                }

            solution = sp.solve(expr, symbols[0])

            return {
                "latex": latex,
                "status": "solved",
                "solution": solution,
                "message": "Equation solved successfully."
            }

        except Exception as e:
            return {
                "latex": latex,
                "status": "not_solved",
                "solution": None,
                "message": str(e)
            }

    # Normal expression
    return {
        "latex": latex,
        "status": "expression",
        "solution": validation["expression"],
        "message": "Valid mathematical expression."
    }

In [ ]:
result = mathink_solve(predicted_latex)


print("LaTeX:", result["latex"])
print("Status:", result["status"])
print("Message:", result["message"])
print("Solution:", result["solution"])

In [ ]:
def classify_expression(latex):

    try:
        # Remove spaces
        latex = latex.replace(" ", "")

        if "=" in latex:

            # Split equation
            lhs_text, rhs_text = latex.split("=", 1)

            # Convert both sides separately
            lhs = latex2sympy(lhs_text)
            rhs = latex2sympy(rhs_text)

            # Move everything to one side
            equation = sp.expand(lhs - rhs)

            variables = list(equation.free_symbols)

            if len(variables) == 0:
                return "constant equation"

            var = variables[0]

            # Polynomial degree
            degree = sp.degree(equation, var)

            if degree == 1:
                return "linear equation"

            elif degree == 2:
                return "quadratic equation"

            elif degree == 3:
                return "cubic equation"

            elif degree is not None and degree > 3:
                return "polynomial equation"

            else:
                return "non-polynomial equation"


        expr = latex2sympy(latex)

        variables = list(expr.free_symbols)

        if len(variables) == 0:
            return "arithmetic expression"

        if expr.is_polynomial(*variables):
            return "polynomial expression"

        return "mathematical expression"

    except Exception as e:

        print("Classification error:", e)

        return "unknown"

In [ ]:
test_expression=[
    "x+5=10",
    "x^2+5*x+6=0",
    "2+3",
    "x^3-2*x=5",
    "x^4+2*x^2=10"
]
for latex in test_expressions:

    category = classify_expression(latex)

    print(f"{latex:25} → {category}")

In [ ]:
# ============================================
# MathInk.ai - Fixed Complete Math Processing
# ============================================

def process_mathink_expression(latex):

    # Make sure input is a string
    if not isinstance(latex, str):
        return {
            "latex": latex,
            "type": "invalid",
            "status": "invalid",
            "solution": None,
            "message": f"Expected LaTeX string, got {type(latex).__name__}"
        }

    # Step 1: Classification
    expression_type = classify_expression(latex)

    # Step 2: Parse the LaTeX
    try:

        # Equations need to be handled by splitting LHS/RHS
        if "=" in latex:

            lhs_text, rhs_text = latex.split("=", 1)

            lhs = latex2sympy(lhs_text)
            rhs = latex2sympy(rhs_text)

            # Convert parsed lists to expressions if necessary
            if isinstance(lhs, list):
                lhs = lhs[0]

            if isinstance(rhs, list):
                rhs = rhs[0]

            equation = sp.Eq(lhs, rhs)

            # Find variables
            variables = list(equation.free_symbols)

            if len(variables) == 0:
                return {
                    "latex": latex,
                    "type": expression_type,
                    "status": "valid",
                    "solution": equation,
                    "message": "Constant equation."
                }

            # Solve
            solution = sp.solve(equation, variables[0])

            return {
                "latex": latex,
                "type": expression_type,
                "status": "solved",
                "solution": solution,
                "message": "Equation solved successfully."
            }

        # --------------------------------
        # Normal expression
        # --------------------------------

        expr = latex2sympy(latex)

        if isinstance(expr, list):
            expr = expr[0]

        simplified = sp.simplify(expr)

        return {
            "latex": latex,
            "type": expression_type,
            "status": "expression",
            "solution": simplified,
            "message": "Expression simplified successfully."
        }

    except Exception as e:

        return {
            "latex": latex,
            "type": expression_type,
            "status": "not_solved",
            "solution": None,
            "message": str(e)
        }

In [ ]:
test_expressions = [
    r"x + 5 = 12",
    r"x^2 + 5*x + 6 = 0",
    r"2 + 3",
    r"x^2 - 9 = 0"
]

for latex in test_expressions:

    result = process_mathink_expression(latex)

    print("=" * 50)
    print("LaTeX      :", result["latex"])
    print("Type       :", result["type"])
    print("Status     :", result["status"])
    print("Solution   :", result["solution"])
    print("Message    :", result["message"])

In [ ]:
# ============================================
# MathInk.ai - Complete Model Pipeline
# ============================================

i = 0

# 1. Get predicted tokens from the model
predicted_tokens = decoded_sequences_100[i]

# 2. Convert tokens → LaTeX
predicted_latex = tokens_to_latex(predicted_tokens)

# 3. Send prediction to Math Engine
result = process_mathink_expression(predicted_latex)

# 4. Display complete result
print("=" * 60)
print("              MathInk.ai RESULT")
print("=" * 60)

print("\nPredicted Tokens:")
print(predicted_tokens)

print("\nPredicted LaTeX:")
print(predicted_latex)

print("\nExpression Type:")
print(result["type"])

print("\nStatus:")
print(result["status"])

print("\nSolution:")
print(result["solution"])

print("\nMessage:")
print(result["message"])

In [ ]:
# ============================================
# Test Multiple MathInk Predictions
# ============================================

num_samples = min(10, len(decoded_sequences_100))

for i in range(num_samples):

    predicted_tokens = decoded_sequences_100[i]

    predicted_latex = tokens_to_latex(predicted_tokens)

    result = process_mathink_expression(predicted_latex)

    print("\n" + "=" * 70)
    print(f"Sample {i + 1}")
    print("=" * 70)

    print("Tokens :", predicted_tokens)
    print("LaTeX  :", predicted_latex)
    print("Type   :", result["type"])
    print("Status :", result["status"])
    print("Result :", result["solution"])

In [ ]:
def step_by_step_solution(latex):

    try:
        # Parse equation
        if "=" not in latex:
            expr = latex2sympy(latex)

            simplified = sp.simplify(expr)

            return {
                "type": "expression",
                "steps": [
                    f"Original expression: {latex}",
                    f"Simplified expression: {simplified}"
                ],
                "answer": simplified
            }

        # Split LHS and RHS
        lhs_text, rhs_text = latex.split("=", 1)

        lhs = latex2sympy(lhs_text)
        rhs = latex2sympy(rhs_text)

        equation = sp.Eq(lhs, rhs)

        variables = list(equation.free_symbols)

        if not variables:
            return {
                "type": "constant equation",
                "steps": [
                    f"Equation: {equation}",
                    f"Result: {sp.simplify(lhs - rhs)}"
                ],
                "answer": sp.simplify(lhs - rhs)
            }

        var = variables[0]

        # Move everything to the left
        standard_form = sp.expand(lhs - rhs)

        degree = sp.degree(standard_form, var)

      

        if degree == 1:

            solution = sp.solve(equation, var)

            steps = [
                f"Given equation: {sp.latex(equation)}",
                f"Move all terms to one side:",
                f"{sp.latex(standard_form)} = 0",
                f"Solve for {var}:",
                f"{var} = {sp.latex(solution[0])}",
                f"Final Answer: {var} = {sp.latex(solution[0])}"
            ]

            return {
                "type": "linear equation",
                "steps": steps,
                "answer": solution
            }

  

        elif degree == 2:

            solution = sp.solve(equation, var)

            factorized = sp.factor(standard_form)

            steps = [
                f"Given equation: {sp.latex(equation)}",
                f"Move all terms to one side:",
                f"{sp.latex(standard_form)} = 0",
                f"Factorize:",
                f"{sp.latex(factorized)} = 0",
                f"Solve for {var}:",
            ]

            for sol in solution:
                steps.append(
                    f"{var} = {sp.latex(sol)}"
                )

            steps.append(
                "Final Answer: "
                + ", ".join(
                    f"{var} = {sp.latex(sol)}"
                    for sol in solution
                )
            )

            return {
                "type": "quadratic equation",
                "steps": steps,
                "answer": solution
            }

        # ====================================
        # OTHER POLYNOMIAL EQUATIONS
        # ====================================

        else:

            solution = sp.solve(equation, var)

            steps = [
                f"Given equation: {sp.latex(equation)}",
                f"Standard form:",
                f"{sp.latex(standard_form)} = 0",
                f"Solving for {var}:",
                f"Solutions: {sp.latex(solution)}"
            ]

            return {
                "type": "polynomial equation",
                "steps": steps,
                "answer": solution
            }

    except Exception as e:

        return {
            "type": "error",
            "steps": [f"Unable to generate steps: {e}"],
            "answer": None
        }

In [ ]:
result = step_by_step_solution(r"x + 5 = 12")

print("Type:", result["type"])
print()

for step in result["steps"]:
    print(step)

print()
print("Answer:", result["answer"])

In [ ]:
result = step_by_step_solution(r"x^2 + 5*x + 6 = 0")

print("Type:", result["type"])
print()

for step in result["steps"]:
    print(step)

print()
print("Answer:", result["answer"])

In [ ]:
def evaluate_mathink_output(latex):
    result=process_mathink_expression(latex)
    status=result["status"]
    expression_type=result["type"]
    if status in ["invalid","invalid_for_solving","not_solving"]:
        confidence="low"
        message="The mathematical expression could not be reliably interpreted"
    elif status=="solved":
        confidence="high"
        message="Mathematical expression recognized and solved successfully"
    elif status == "expression":
        confidence = "medium"
        message = "Expression recognized successfully, but no equation was solved."

    else:
        confidence = "low"
        message = "Recognition confidence is low."

    return {
        "latex": latex,
        "type": expression_type,
        "status": status,
        "confidence": confidence,
        "solution": result["solution"],
        "message": message
    }

In [ ]:
test_outputs = [
    r"x + 5 = 12",
    r"x^2 + 5*x + 6 = 0",
    r"h(a)=b(b)",
    r"2 + 3"
]

for latex in test_outputs:

    result = evaluate_mathink_output(latex)

    print("=" * 60)
    print("LaTeX      :", result["latex"])
    print("Type       :", result["type"])
    print("Status     :", result["status"])
    print("Confidence :", result["confidence"])
    print("Solution   :", result["solution"])
    print("Message    :", result["message"])

In [ ]:
i=0
predicted_tokens=decoded_sequences_100[i]
predicted_latex=tokens_to_latex(predicted_tokens)
print("Predicted Tokens:")
print(predicted_tokens)

print("\nPredicted LaTeX:")
print(predicted_latex)

result = step_by_step_solution(predicted_latex)

print("\nExpression Type:")
print(result["type"])

print("\nStep-by-Step Solution:")
for number, step in enumerate(result["steps"], 1):
    print(f"{number}. {step}")

print("\nFinal Answer:")
print(result["answer"])

In [ ]:
def create_solution_output(latex):

    result = step_by_step_solution(latex)

    output = {
        "input_latex": latex,
        "expression_type": result["type"],
        "steps": [],
        "final_answer": result["answer"]
    }

    for i, step in enumerate(result["steps"], 1):
        output["steps"].append({
            "step_number": i,
            "explanation": step
        })
    return output

In [ ]:
latex = r"x^2 + 5*x + 6 = 0"

solution_output = create_solution_output(latex)

print("Recognized LaTeX:")
print(solution_output["input_latex"])

print("\nExpression Type:")
print(solution_output["expression_type"])

print("\nSteps:")

for step in solution_output["steps"]:
    print(
        f"Step {step['step_number']}: "
        f"{step['explanation']}"
    )

print("\nFinal Answer:")
print(solution_output["final_answer"])